<a href="https://colab.research.google.com/github/robinacharya/Ensemble-Machine-Learning-Algorithms-for-Real-World-Data-RWD-/blob/GBM/GBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
# Load the dataset
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the dataset from Google Drive
path_to_dataset = "/content/drive/MyDrive/german_credit_data.csv"
df = pd.read_csv(path_to_dataset)

In [ ]:
# Display the first few rows of the dataset to understand its structure
df.head()


In [ ]:
# Encode categorical variables if necessary
label_encoders = {}
for column in df.select_dtypes(include=['object']).columns:
    label_encoders[column] = LabelEncoder()
    df[column] = label_encoders[column].fit_transform(df[column])

In [ ]:
# Define features and target variable
X = df.drop(columns=['Risk'])
y = df['Risk']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Initialize the baseline Gradient Boosting Model
baseline_gbm = GradientBoostingClassifier(random_state=42)
baseline_gbm.fit(X_train, y_train)

# Predict on the test set using the baseline model
y_pred_baseline = baseline_gbm.predict(X_test)

In [ ]:
# Evaluate the performance of the baseline model
print("Baseline GBM Model Performance")
print("Accuracy:", accuracy_score(y_test, y_pred_baseline))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_baseline))
print("Classification Report:\n", classification_report(y_test, y_pred_baseline))

In [ ]:
# Set up the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7, 20],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [4, 8]
}

# Initialize GridSearchCV with the Gradient Boosting Classifier
grid_search = GridSearchCV(estimator=GradientBoostingClassifier(random_state=42),
                           param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Retrieve the best estimator
best_gbm = grid_search.best_estimator_
# Print the best parameters found by GridSearchCV
print("Best Parameters Found by GridSearchCV:\n", grid_search.best_params_)

In [ ]:
# Train the best estimator with the training data
best_gbm.fit(X_train, y_train)

# Predict on the test set using the tuned model
y_pred_tuned = best_gbm.predict(X_test)

In [ ]:
# Evaluate the performance of the tuned model
print("Tuned GBM Model Performance")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_tuned))
print("Classification Report:\n", classification_report(y_test, y_pred_tuned))

In [ ]:
# Import necessary modules
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt # Import matplotlib

# Calculate predicted probabilities for the baseline model (replace 'your_baseline_model' with the actual model object)
y_prob_baseline = baseline_gbm.predict_proba(X_test)[:, 1]  # Assuming binary classification, take probabilities for class 1

# Assuming you have y_pred_baseline and y_prob_baseline from your baseline model evaluation
# If not, you need to calculate them first
fpr_baseline, tpr_baseline, _ = roc_curve(y_test, y_prob_baseline)  # Calculate fpr_baseline, tpr_baseline
roc_auc_baseline = auc(fpr_baseline, tpr_baseline)  # Calculate roc_auc_baseline

y_pred_tuned = best_gbm.predict(X_test)
y_prob_tuned = best_gbm.predict_proba(X_test)[:, 1]

# Evaluate the performance of the tuned model
print("Tuned GBM Model Performance")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_tuned))
print("Classification Report:\n", classification_report(y_test, y_pred_tuned))

# Compute ROC curve and ROC area for tuned model
fpr_tuned, tpr_tuned, _ = roc_curve(y_test, y_prob_tuned)
roc_auc_tuned = auc(fpr_tuned, tpr_tuned)

# Plotting the ROC curves
plt.figure()
plt.plot(fpr_baseline, tpr_baseline, color='darkorange', lw=2, label='Baseline GBM ROC curve (area = %0.2f)' % roc_auc_baseline)
plt.plot(fpr_tuned, tpr_tuned, color='blue', lw=2, label='Tuned GBM ROC curve (area = %0.2f)' % roc_auc_tuned)
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()
